# 05. Реранкер (cross-encoder) и второй этап ранжирования

После `04` нужное объявление есть в пуле в ~99% случаев, но в топ-50 попадает в ~96%: ранжировщик теряет ~3 пункта
на выборе среди похожих кандидатов. Cross-encoder читает запрос и объявление **вместе** и точнее различает
близкие по смыслу объявления («ремонт» vs «продажа», «для ИП» vs «для физлиц»).

1. Дообучение `BAAI/bge-reranker-v2-m3` на train (код — `experiments/train_reranker.py`):
   позитив — выбранное объявление, 4 трудных негатива — самые похожие по e5-large объявления той же локации,
   кроме любых объявлений, которые выбирали по этому же тексту запроса (иначе учили бы ложные негативы).
   40 тыс. запросов, 200 тыс. пар, 1 эпоха, Adafactor, bf16, ~70 мин на RTX 5060 Ti.
2. Скоринг топ-100 кандидатов этапа 1 для валидации (OOF) и бенчмарка.
3. Второй LightGBM: признаки этапа 1 + скор этапа 1 + скор реранкера → топ-50. Оценка — та же 5-fold CV по запросам.

In [1]:
import sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, '..')
from src.pipeline import ART
from src.ranking import cv_ranker, fit_ranker, make_answer, check_answer, recall_by_score
from src.prep import recall_at_k

RETRAIN = False   # True — заново собрать данные, обучить реранкер и посчитать скоры (GPU, ~1.5 ч)
if RETRAIN:
    for stage in ['data', 'train', 'score']:
        subprocess.run([sys.executable, '-m', 'experiments.train_reranker', '--stage', stage, '--top', '100'], cwd='..', check=True)

In [2]:
q_val = pd.read_parquet(ART / 'queries_val.parquet')
q_bench = pd.read_parquet(ART / 'queries_bench.parquet')
corpus_ids = pd.read_parquet(ART / 'corpus.parquet', columns=['item_id', 'in_bench'])

def stage2_frame(pool_file, stage1_file, rerank_file):
    # Кандидаты = топ-100 этапа 1; к ним признаки пула, скор этапа 1 и скор реранкера
    st = pd.read_parquet(ART / stage1_file)
    ce = pd.read_parquet(ART / rerank_file)
    df = ce.merge(st, on=['q_row', 'item_id'], how='left')
    pool = pd.read_parquet(ART / pool_file)
    df = df.merge(pool.drop(columns=[c for c in ['label'] if c in pool and 'label' in df]), on=['q_row', 'item_id'], how='left')
    g = df.groupby('q_row')
    df['s1_rank'] = g['pred'].rank(ascending=False, method='first')
    df['ce_rank'] = g['ce'].rank(ascending=False, method='first')
    df['ce_z'] = (df['ce'] - g['ce'].transform('mean')) / (g['ce'].transform('std') + 1e-6)
    df['ce_gap'] = df['ce'] - g['ce'].transform('max')
    df['s1_z'] = (df['pred'] - g['pred'].transform('mean')) / (g['pred'].transform('std') + 1e-6)
    return df.rename(columns={'pred': 's1_pred'}).sort_values('q_row', kind='stable').reset_index(drop=True)

val2 = stage2_frame('pool_val.parquet', 'stage1_oof_val.parquet', 'rerank_val.parquet')
bench2 = stage2_frame('pool_bench.parquet', 'stage1_bench.parquet', 'rerank_bench.parquet')
print('val:', val2.shape, '| bench:', bench2.shape)
print('Recall@100 этапа 1 (потолок этапа 2):', round(val2.groupby('q_row')['label'].sum().reindex(range(len(q_val))).fillna(0).div(q_val['relevant'].str.len()).mean(), 4))

val: (600000, 68) | bench: (245200, 67)
Recall@100 этапа 1 (потолок этапа 2): 0.9766


In [3]:
res = {
    'этап 1 (LightGBM из 04)': recall_by_score(val2, q_val, val2['s1_pred']),
    'только реранкер': recall_by_score(val2, q_val, val2['ce']),
}
for w in [0.3, 0.5, 1.0]:
    res[f'смесь z(этап1) + {w}·z(реранкер)'] = recall_by_score(val2, q_val, val2['s1_z'] + w * val2['ce_z'])
pd.Series(res, name='Recall@50').round(4)

этап 1 (LightGBM из 04)             0.9585
только реранкер                     0.7849
смесь z(этап1) + 0.3·z(реранкер)    0.9582
смесь z(этап1) + 0.5·z(реранкер)    0.9561
смесь z(этап1) + 1.0·z(реранкер)    0.9508
Name: Recall@50, dtype: float64

In [4]:
rec2, it2, oof2 = cv_ranker(val2, q_val, rounds=2000)
print(f'Этап 2, OOF Recall@50 = {rec2:.4f} (итераций {it2})')
qv = q_val.reset_index(drop=True)
top = oof2.sort_values(['q_row', 'pred'], ascending=[True, False]).groupby('q_row').head(50)
hit = top[top['label'] == 1].groupby('q_row').size().reindex(range(len(qv))).fillna(0) / qv['relevant'].str.len()
print(hit.groupby(qv['bucket']).mean().round(4).to_dict())
in_bench = set(corpus_ids.loc[corpus_ids['in_bench'] == 1, 'item_id'])
mask = qv['relevant'].map(lambda r: all(x in in_bench for x in r)).values
print(f'Контрольная группа (позитивы из корпуса бенчмарка): {hit[mask].mean():.4f}; остальные: {hit[~mask].mean():.4f}')

Этап 2, OOF Recall@50 = 0.9583 (итераций 47)


{'1': 0.9575, '2-5': 0.9556, '6-20': 0.9699, '>20': 0.9901, 'unseen': 0.9552}
Контрольная группа (позитивы из корпуса бенчмарка): 0.9306; остальные: 0.9604


## Ответ

Финальная модель — среднее 5 LightGBM (разные seed), обученных на всех отложенных поисках.

In [5]:
score = np.zeros(len(bench2))
for seed in range(5):
    m, cols = fit_ranker(val2, it2, seed=seed)
    s = m.predict(bench2[cols])
    score += (s - s.mean()) / (s.std() + 1e-9)
answer = make_answer(bench2, q_bench, score, k=50)
check_answer(answer, q_bench, corpus_ids.loc[corpus_ids['in_bench'] == 1, 'item_id'])
answer.to_csv('../answer_stage2.csv', index=False)
print(answer.shape, answer['answer'].str.split().str.len().min())

(2452, 2) 50
